In [1]:
! pip install ultralytics supervision roboflow inference

In [2]:
import os

os.getcwd()

'/content'

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/Projects/Football_Analysis/

/content/drive/MyDrive/Projects/Football_Analysis


In [7]:
import numpy as np
import json
import os

from utils import read_video, save_video
from trackers import Tracker
from team_color_assigner import TeamColorAssigner
from player_ball_assigner import PlayerBallAssigner
from camera_movement_estimator import CameraMovementEstimator
from pitch_key_point import PitchKeyPoints , SoccerPitchConfiguration
from view_transformer import ViewTransformer
from pitch_key_point.draw_pitch import draw_pitch, draw_points_on_pitch


def main():

    # Read the video file
    video_frames = read_video('input_video/08fd33_4.mp4')

    # Initialize tracker
    tracker = Tracker("models/player_detection/best.pt")

    # Get object tracks
    tracks = tracker.get_object_tracks(video_frames,
                                       read_from_stubs= True,
                                       stub_path='stubs/track_stubs.pkl')

    # Get object position
    tracker.add_position_to_tracks(tracks)

    # Camera movement estimator
    camera_movement_estimator = CameraMovementEstimator(video_frames[0])
    camera_movement_per_frame = camera_movement_estimator.get_camera_movement(video_frames,
                                                                              read_from_stub=True,
                                                                              stub_path='stubs/camera_movement_stub.pkl')
    camera_movement_estimator.add_adjust_positions_to_tracks(tracks, camera_movement_per_frame)

    # Interpolate ball positions
    tracks['ball'] = tracker.interpolate_ball_positions(tracks['ball'])

    # Assign player teams
    team_assigner = TeamColorAssigner()
    team_assigner.assign_team_color(video_frames[0], tracks['players'][0])

    for frame_num, player_track in enumerate(tracks['players']):
        for player_id, track in player_track.items():
            team = team_assigner.get_player_team(video_frames[frame_num],
                                                 track['bbox'],
                                                 player_id)
            tracks['players'][frame_num][player_id]['team'] = team
            tracks['players'][frame_num][player_id]['team_color'] = team_assigner.team_colors[team]

    # Assign ball acquisition
    player_assigner = PlayerBallAssigner()
    team_ball_control = []

    for frame_num, player_track in enumerate(tracks['players']):
        ball_bbox = tracks['ball'][frame_num][1]['bbox']
        assigned_player = player_assigner.assign_ball_to_player(player_track, ball_bbox)

        if assigned_player != -1:
            tracks['players'][frame_num][assigned_player]['has_ball'] = True
            team_ball_control.append(tracks['players'][frame_num][assigned_player]['team'])
        else:
            team_ball_control.append(team_ball_control[-1])

    team_ball_control = np.array(team_ball_control)

    # Draw output
    ## Draw object tracks
    output_video_frames = tracker.draw_annotations(video_frames, tracks, team_ball_control)

    # Draw camera movement
    output_video_frames = camera_movement_estimator.draw_camera_movement(output_video_frames, camera_movement_per_frame)

    # Pitch keypoint detection
    path = os.getcwd()
    api_key_path = os.path.join(path, 'training/Roboflow.json')
    roboflow_api = json.load(open(api_key_path))
    api_key = roboflow_api["api_key"]

    pitch_key_point = PitchKeyPoints()
    frame_points, pitch_points = pitch_key_point.key_point_detection(video_frames, api_key)

    CONFIG = SoccerPitchConfiguration()
    output_radar_frames = []

    for frame_num, frame in enumerate(video_frames):

        radar = draw_pitch(config=CONFIG)
        transformer = ViewTransformer(
        source=frame_points[frame_num].astype(np.float32),
        target=pitch_points[frame_num].astype(np.float32)
        )

        output_radar_frames.append(radar)


    # Save the video file
    save_video(output_video_frames, 'output_videos/output_video.avi')


In [8]:
main()

AttributeError: 'KeyPoints' object has no attribute 'astype'